最初版，导入MCC ESD的表，任何带NA值的row直接被去掉。权重正弦推荐模型。

In [ ]:
import pandas as pd

df=pd.read_csv(
    'MCC_product_lists/MCC_DataExport_esd-protection-devices(MCC-esd-protection-devices).csv',
    header=0,
    skiprows=[0,2]
)

In [81]:
df.head()

,Manufacture,Product,Status,Compliance,Number of Functions,Configuration,Package Type,Reverse\nStandoff\nVoltage\nVRWM(V),Peak Pulse\nCurrent\nIPP(A),Max.\nClamping\nVoltage\nVC (V),Junction\nCapacitance\nCJ(pF),Peak Pluse\nPower\nDissipation\nPPPK (W),Maximum\nReverse\nLeakage\nIR (uA),Breakdown\nVoltage\nMin VBR(V),Breakdown\nVoltage\nMax VBR(V),Junction\nTemperature\nTj [max] (°C),VESDIEC61000-4-2\nAir/Contact\n(kV)
0,MCC (Micro Commercial Components),MMBZ12VAQ,Preferred,A R H,2,Unidirectional,SOT-23,12.0,2.35,17.0,113.00,40.0,0.20,11.40,12.60,150,±30
1,MCC (Micro Commercial Components),MMBZ5V6AQ,Preferred,A R H,2,Unidirectional,SOT-23,5.6,3.00,8.0,320.00,24.0,5.00,5.32,5.88,150,±30
2,MCC (Micro Commercial Components),MMBZ9V1AQ,Preferred,A R H,2,Unidirectional,SOT-23,9.1,1.70,14.0,183.00,24.0,0.30,8.65,9.56,150,±30
3,MCC (Micro Commercial Components),ESDSBULC24VLBQ,Preferred,A R H,1,Bidirectional,DFN1006-2,24.0,4.00,9.0,0.45,36.0,0.05,25.00,32.00,150,±15
4,MCC (Micro Commercial Components),ESDSBULC18VLBQ,Preferred,A R H,1,Bidirectional,DFN1006-2,18.0,4.00,9.0,0.45,36.0,0.05,18.50,NaN,150,±15


In [82]:
df.columns = [' '.join(col.split()) for col in df.columns]
df.columns.tolist()

['Manufacture',
 'Product',
 'Status',
 'Compliance',
 'Number of Functions',
 'Configuration',
 'Package Type',
 'Reverse Standoff Voltage VRWM(V)',
 'Peak Pulse Current IPP(A)',
 'Max. Clamping Voltage VC (V)',
 'Junction Capacitance CJ(pF)',
 'Peak Pluse Power Dissipation PPPK (W)',
 'Maximum Reverse Leakage IR (uA)',
 'Breakdown Voltage Min VBR(V)',
 'Breakdown Voltage Max VBR(V)',
 'Junction Temperature Tj [max] (°C)',
 'VESDIEC61000-4-2 Air/Contact (kV)']

In [83]:
df_selected=df.iloc[:,[1]+list(range(4,17))]
# df_selected=df.iloc[:,[1]+list(range(4,16))]
df_selected.head()

,Product,Number of Functions,Configuration,Package Type,Reverse Standoff Voltage VRWM(V),Peak Pulse Current IPP(A),Max. Clamping Voltage VC (V),Junction Capacitance CJ(pF),Peak Pluse Power Dissipation PPPK (W),Maximum Reverse Leakage IR (uA),Breakdown Voltage Min VBR(V),Breakdown Voltage Max VBR(V),Junction Temperature Tj [max] (°C),VESDIEC61000-4-2 Air/Contact (kV)
0,MMBZ12VAQ,2,Unidirectional,SOT-23,12.0,2.35,17.0,113.00,40.0,0.20,11.40,12.60,150,±30
1,MMBZ5V6AQ,2,Unidirectional,SOT-23,5.6,3.00,8.0,320.00,24.0,5.00,5.32,5.88,150,±30
2,MMBZ9V1AQ,2,Unidirectional,SOT-23,9.1,1.70,14.0,183.00,24.0,0.30,8.65,9.56,150,±30
3,ESDSBULC24VLBQ,1,Bidirectional,DFN1006-2,24.0,4.00,9.0,0.45,36.0,0.05,25.00,32.00,150,±15
4,ESDSBULC18VLBQ,1,Bidirectional,DFN1006-2,18.0,4.00,9.0,0.45,36.0,0.05,18.50,NaN,150,±15


In [84]:
df_selected.info()

<class 'pandas.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 14 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Product                                131 non-null    str    
 1   Number of Functions                    131 non-null    int64  
 2   Configuration                          131 non-null    str    
 3   Package Type                           131 non-null    str    
 4   Reverse Standoff Voltage VRWM(V)       131 non-null    float64
 5   Peak Pulse Current IPP(A)              131 non-null    float64
 6   Max. Clamping Voltage VC (V)           131 non-null    float64
 7   Junction Capacitance CJ(pF)            127 non-null    float64
 8   Peak Pluse Power Dissipation PPPK (W)  131 non-null    float64
 9   Maximum Reverse Leakage IR (uA)        131 non-null    float64
 10  Breakdown Voltage Min VBR(V)           131 non-null    float64
 11  Breakdown Voltage

In [85]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

def apply_hard_constraints(df, query_chip):
    # 从 query_chip 中提取标量值（假设只有一行）
    num_functions = query_chip['Number of Functions'].iloc[0]
    config = query_chip['Configuration'].iloc[0]
    package = query_chip['Package Type'].iloc[0]
    
    filtered = df[
        (df['Number of Functions'] == num_functions) &
        (df['Configuration'] == config) &
        (df['Package Type'] == package)
    ].copy()
    
    return filtered

In [86]:
columns=df_selected.columns.tolist()

# this one works
# values=[
#     'ESD2CANFD24T2Q',
#     2,
#     'Bidirectional',
#     'SOT-23',
#     24,
#     3.5,
#     36,
#     2.5,
#     133,
#     0.05,
#     25.5,
#     35.5,
#     150,
#     '±25'
# ]

# this one does not work
values=[
    'NUP2105LT1G',
    2,
    'Bidirectional',
    'SOT-23',
    24,
    8.0,
    44,
    30,
    350,
    0.1,
    26.2,
    32,
    150,
    '±30'
]
query_chip=pd.DataFrame([values],columns=columns)
query_chip.head()

,Product,Number of Functions,Configuration,Package Type,Reverse Standoff Voltage VRWM(V),Peak Pulse Current IPP(A),Max. Clamping Voltage VC (V),Junction Capacitance CJ(pF),Peak Pluse Power Dissipation PPPK (W),Maximum Reverse Leakage IR (uA),Breakdown Voltage Min VBR(V),Breakdown Voltage Max VBR(V),Junction Temperature Tj [max] (°C),VESDIEC61000-4-2 Air/Contact (kV)
0,NUP2105LT1G,2,Bidirectional,SOT-23,24,8.0,44,30,350,0.1,26.2,32,150,±30


In [87]:
# query_data={
#     'Product':['ESD2CANFD24T2Q'],
#     'Number of Functions':[2],
#     'Configuration':['Bidirectional'],
#     'Package Type':['SOT-23'],
#     'Reverse Standoff Voltage VRWM(V)':[24],
#     'Peak Pulse Current IPP(A)':[3.5],
#     'Max. Clamping Voltage VC (V)':[36],
#     'Junction Capacitance CJ(pF)':[2.5],
#     'Peak Pluse Power Dissipation PPPK (W)':[133],
#     'Maximum Reverse Leakage IR (uA)':[0.05],
#     'Breakdown Voltage Min VBR(V)':[25.5],
#     'Breakdown Voltage Max VBR(V)':[35.5],
#     'Junction Temperature Tj [max] (°C)':[150],
#     'VESDIEC61000-4-2 Air/Contact (kV)':['±25']
# }
# query_chip = pd.DataFrame(query_data)
# query_chip.head()

In [88]:
# 定义特征列
# 方法：从 df_selected 中排除不需要的列，剩下的就是数值特征
# exclude_columns = ['Product', 'Number of Functions', 'Configuration', 'Package Type', 'VESDIEC61000-4-2 Air/Contact (kV)']
# numeric_features = [col for col in df_selected.columns if col not in exclude_columns]
# categorical_features = ['VESDIEC61000-4-2 Air/Contact (kV)']

numeric_features = ['Reverse Standoff Voltage VRWM(V)',
 'Peak Pulse Current IPP(A)',
 'Max. Clamping Voltage VC (V)',
 'Junction Capacitance CJ(pF)',
 'Peak Pluse Power Dissipation PPPK (W)',
 'Maximum Reverse Leakage IR (uA)',
 'Breakdown Voltage Min VBR(V)',
 'Breakdown Voltage Max VBR(V)',
 'Junction Temperature Tj [max] (°C)',]
categorical_features = ['VESDIEC61000-4-2 Air/Contact (kV)']

# 创建预处理流水线
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),           # 数值型：标准化
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)  # 类别型：独热编码
])
df_selected=df_selected.dropna()
preprocessor.fit(df_selected)

# 把所有特征拼成一个向量
def encode_chip(df_chips):
    return preprocessor.transform(df_chips)

# # # 把客户芯片也编码成同样的格式
# query_vector = encode_chip(query_chip)
# candidate_vectors = encode_chip(df_selected)

In [89]:
import numpy as np

def weighted_similarity(query_vec, candidate_matrix, feature_weights):
    """
    计算加权余弦相似度
    feature_weights: 列表，顺序与编码后的特征列顺序一致
    """
    # 先对候选矩阵的每一列乘以对应的权重
    weighted_candidates = candidate_matrix * feature_weights
    weighted_query = query_vec * feature_weights
    
    # 计算余弦相似度
    similarities = cosine_similarity(weighted_query, weighted_candidates)
    return similarities.flatten()

# 假设你的特征编码后顺序是：[vdd_min, vdd_max, i_max, temp_range, brand_TI, brand_ADI, brand_NXP]
# 权重设：电压最重要(2.0)，电流次之(1.5)，温度(1.0)，品牌(0.5)
# feature_weights = np.array([1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])

In [90]:
def recommend_alternatives(df_inventory, query_chip, top_k=10, weights=None):
    """
    输入客户芯片参数，返回最相似的可替代芯片列表
    """
    # 1. 硬约束过滤
    candidates = apply_hard_constraints(df_inventory, query_chip)
    if candidates.empty:
        return "没有引脚/封装完全匹配的替代品"
    
    # 2. 编码
    query_vec = encode_chip(query_chip)
    candidate_vecs = encode_chip(candidates)
    
    # 3. 计算相似度
    if weights is None:
        # 默认全部权重为1
        weights = np.ones(candidate_vecs.shape[1])
    scores = weighted_similarity(query_vec, candidate_vecs, weights)
    
    # 4. 排序并返回
    candidates['similarity_score'] = scores
    results = candidates.sort_values('similarity_score', ascending=False).head(top_k)
    return_results = results[['Product', 'similarity_score'] + numeric_features + categorical_features]
    return return_results

# 使用示例
result = recommend_alternatives(df_selected, query_chip, top_k=3)
print(result)

         Product  similarity_score  Reverse Standoff Voltage VRWM(V)  \
87     HSM24BHE3          0.991662                              24.0   
86  ESD24VT2BHE3          0.988134                              24.0   
82  ESD27VT2BHE3          0.892285                              27.0   

    Peak Pulse Current IPP(A)  Max. Clamping Voltage VC (V)  \
87                        8.0                          48.0   
86                        7.0                          44.0   
82                        6.0                          70.0   

    Junction Capacitance CJ(pF)  Peak Pluse Power Dissipation PPPK (W)  \
87                         32.0                                  384.0   
86                         30.0                                  308.0   
82                         13.0                                  420.0   

    Maximum Reverse Leakage IR (uA)  Breakdown Voltage Min VBR(V)  \
87                              0.1                          26.3   
86                     